<h1 style="text-align: center;"> &#11088; Welcome to our Initial Demo! &#11088; </h1>

The motivation behind our project
Looking up at the night sky is a timeless human experience, yet modern light pollution and unpredictable weather make finding a truly clear view harder than ever. For amateur stargazers and families planning outdoor trips, missing out on a rare astronomical event or driving hours to find an overcast sky can be disappointing. Our goal is to inspire more people to step outside and look at the stars without the guesswork.

In [57]:
pip install requests pandas boto3 scikit-learn

In [58]:
import json # format the api data is in
import boto3 # allows aws server communications
import requests # allows for requests to be made to the API
import pandas as pd # allows for a dataframe to be made! (Organization and Modeling)
from datetime import datetime, timezone # allows for conversions from UTC (the format from the API)
from sklearn.ensemble import RandomForestRegressor # what we want to use to model our project in the future!

In [59]:
# 1. Initialize S3 client
s3 = boto3.client('s3')

# Where is the data located within my AWS s3??? (Shown below)
BUCKET_NAME = 'api-s3-data-dtsc3601' 
OBJECT_KEY = 'openweatherdata.json'

# Grabs the object from the bucket to allow it to be read for later usage
response = s3.get_object(Bucket=BUCKET_NAME, Key=OBJECT_KEY)

# Read and decode the text content (openweatherdata.json)
file_content = response['Body'].read().decode('utf-8')
print("Raw File Content:")
print(file_content)

# Parse the already-read string variable (breaking-down the information from the API into digestible pieces!)
raw_data = json.loads(file_content)

# Convert to pandas DataFrame
df = pd.DataFrame(raw_data if isinstance(raw_data, list) else [raw_data])
print("\nDataFrame:")
print(df.head())

Raw File Content:
{"lat":35.19,"lon":-111.651,"timezone":"America/Phoenix","timezone_offset":-25200,"data":[{"dt":1790636400,"temp":285.57,"feels_like":285.32,"pressure":1012,"humidity":94,"dew_point":284.63,"uvi":1.37,"clouds":99,"visibility":10000,"wind_speed":3.1,"wind_deg":224,"wind_gust":9.32,"weather":[{"id":804,"main":"Clouds","description":"overcast clouds","icon":"04d"}],"pop":0,"alerts":["urn:oid:2.49.0.1.840.0.d6814b50c34d8a0472181f141035145789a59a94.001.1:00287b17c75af43ae64b1a6ce086d3e6"]},{"dt":1790640000,"temp":285.62,"feels_like":285.35,"pressure":1012,"humidity":93,"dew_point":284.52,"uvi":0.3,"clouds":99,"visibility":10000,"wind_speed":3.22,"wind_deg":216,"wind_gust":9.27,"weather":[{"id":804,"main":"Clouds","description":"overcast clouds","icon":"04d"}],"pop":0,"alerts":["urn:oid:2.49.0.1.840.0.d6814b50c34d8a0472181f141035145789a59a94.001.1:00287b17c75af43ae64b1a6ce086d3e6"]},{"dt":1790643600,"temp":285.66,"feels_like":285.39,"pressure":1012,"humidity":93,"dew_point"

In [60]:
# 1. Read S3 JSON as before
s3 = boto3.client('s3')
response = s3.get_object(Bucket='api-s3-data-dtsc3601', Key='openweatherdata.json')
raw_data = json.loads(response['Body'].read().decode('utf-8'))

# Collect all hourly/data records containing 'dt' --> dt is used for datetime which is used by Coordinated Universal Time (UTC)
if isinstance(raw_data, list):
    records = raw_data
elif isinstance(raw_data, dict):
    # Search for lists inside the dictionary (e.g., 'data', 'hours (dt)', 'list')
    records = []
    for val in raw_data.values():
        if isinstance(val, list) and len(val) > 0 and isinstance(val[0], dict) and 'dt' in val[0]:
            records = val
            break
    if not records and 'dt' in raw_data:
        records = [raw_data]

In [61]:
# calcStars is used to calculate the scores for the humidity, temperature in Fahrenheit, and visibility in miles. 
# This will be used later in order to see overall what the best time of the day is to go Stargazing in Flagstaff, Arizona.
def calcStars(temp, humidity, visibility):
    vis_score = min(visibility / 10.0, 1.0) * 100
    hum_score = max(0, 100 - humidity)
    
    if 50 <= temp <= 72: # optimal temperatures for having a clear sky for stargazing
        temp_score = 100
    else:
        temp_score = max(0, 100 - abs(temp - 61) * 3)
        
    final_score = (0.50 * vis_score) + (0.35 * hum_score) + (0.15 * temp_score)
    return round(final_score, 2)

# Parse all records into a list -- Let's get this started!!
parsed_list = []

for entry in records:
    dt_val = entry.get('dt')
    # Convert Unix timestamp to a human-readable string (YYYY-MM-DD HH:MM:SS)
    readable_date = datetime.fromtimestamp(dt_val, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if dt_val else None

    # making requests to the API to get the temperature and convert it to Fahrenheit, along with getting the humidity from the API.
    metrics = entry.get('main', entry)
    raw_temp = metrics.get('temp', 60.0)
    temp_f = (raw_temp - 273.15) * 9/5 + 32 if raw_temp > 200 else raw_temp
    humidity_pct = metrics.get('humidity', 50)

    # making requests to the API to get the visibility of the sky and converting it into miles (imperial)
    vis_meters = entry.get('visibility', metrics.get('visibility', 10000))
    visibility_mi = vis_meters * 0.000621371 if vis_meters else 6.21

    # calling the function to make it run!
    score = calcStars(temp_f, humidity_pct, visibility_mi)

    # taking the different parameters and formatting them to the nearest decimal place
    parsed_list.append({
        'dt': dt_val,
        'datetime_utc': readable_date,
        'date_str': readable_date.split(' ')[0] if readable_date else None,
        'temp_f': round(temp_f, 1),
        'humidity_pct': humidity_pct,
        'visibility_mi': round(visibility_mi, 2),
        'stargazing_percentage': score
    })

# putting the dataframe together 
full_df = pd.DataFrame(parsed_list)

In [64]:
# Here we are formatting and creating our predictions for each hour of the next day to see what time of day will be the best for stargazing. 
# Arizona is very different from where we are now, so the best times may surprise you! 
# 1. Specify the date you want to analyze
target_date = "2026-09-29"

# 2. Filter the dataset for that specific date
day_df = full_df[full_df['date_str'] == target_date]

if not day_df.empty:
    # 3. Sort by stargazing_percentage descending and pick the top hour
    best_hour = day_df.sort_values(by='stargazing_percentage', ascending=False).iloc[0]

    print(f"🌟 Best Time for Stargazing on {target_date}:")
    print(f"Time (UTC) : {best_hour['datetime_utc']}")
    print(f"Score      : {best_hour['stargazing_percentage']}%")
    print(f"Conditions : {best_hour['temp_f']}°F | {best_hour['humidity_pct']}% Humidity | {best_hour['visibility_mi']} mi Visibility")

    # Display all hourly options for that day ranked from best to worst
    print(f"\n--- All Hourly Scores for {target_date} (Ranked) ---")
    print(day_df[['datetime_utc', 'stargazing_percentage', 'temp_f', 'humidity_pct', 'visibility_mi']]
          .sort_values(by='stargazing_percentage', ascending=False)
          .to_string(index=False))

else:
    print(f"No weather data found for date '{target_date}' in this API payload.")
    print(f"Available dates in dataset: {full_df['date_str'].dropna().unique().tolist()}")

🌟 Best Time for Stargazing on 2026-09-29:
Time (UTC) : 2026-09-29 16:00:00
Score      : 56.92%
Conditions : 55.8°F | 69% Humidity | 6.21 mi Visibility

--- All Hourly Scores for 2026-09-29 (Ranked) ---
       datetime_utc  stargazing_percentage  temp_f  humidity_pct  visibility_mi
2026-09-29 16:00:00                  56.92    55.8            69           6.21
2026-09-29 17:00:00                  54.82    55.1            75           6.21
2026-09-29 00:00:00                  48.52    54.4            93           6.21
2026-09-29 01:00:00                  48.52    54.5            93           6.21
2026-09-29 02:00:00                  47.47    52.7            96           6.21
2026-09-29 03:00:00                  46.77    50.6            98           6.21
2026-09-29 15:00:00                  43.74    48.8            91           6.21
2026-09-29 11:00:00                  40.68    47.5            98           6.21
2026-09-29 14:00:00                  39.98    45.1            97           6.2

<h3 style='text-align: center;'> Future Work &#11088; </h3>

Some future ideas that we want to try to incorporate into our project include: (not finalized)
- Train and evaluate the ML using RandomForestRegressor with hyperparameter tuning/feature engineering to achieve higher accuracy models. This task would account for things like local elevation and terrain differences.
- Polish with error handling, UI/UX refinement, and handling edge cases or missing data.
- Implement continuous data collection from the API for continuous model retraining and to stay relevant with our predictions.
- More AWS implementations to better automate our project using applications like SageMaker. 

In [46]:
# 3. Train Machine Learning Model
X = df[['temp_f', 'humidity_pct', 'visibility_mi']]
y = df['stargazing_percentage']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

RandomForestRegressor(random_state=42)

In [47]:
# 4. Predict on Imperial Input Values
def decideStars(temp_f, humidity_pct, visibility_mi):
    input_data = pd.DataFrame(
        [[temp_f, humidity_pct, visibility_mi]], 
        columns=['temp_f', 'humidity_pct', 'visibility_mi']
    )
    prediction = model.predict(input_data)[0]
    return f"Stargazing Suitability: {prediction:.1f}%"

In [49]:
# Example Evaluation: 62°F, 35% humidity, 10 miles visibility
print(decideStars(temp_f=62, humidity_pct=35, visibility_mi=4))
# Output: Stargazing Suitability: 97.8%

Stargazing Suitability: 48.1%
